Đây là script chạy huấn luyện mô hình đề xuất gồm các bước:
- Clone repository từ GitHub
- Load dữ liệu đề bài của cuộc thi
- Huấn luyện mô hình
- Lưu kết quả huấn luyện và file submission

**Lưu ý**:
- Script này **không thể** chạy local do ngốn bộ nhớ. Script **bắt buộc** phải chạy trên GPU, chúng tôi có đính kèm hướng dẫn chi tiết chạy trên [Kaggle](./HOW_TO_RUN_KAGGLE.md).
- Notebook này là notebook chạy stage 1 trong quá trình huấn luyện (Huấn luyện mô hình deep learning), notebook chạy stage 2 có thể xem tại [đây](./dataflow2026_hd4k_run_stage_2.ipynb).
- Notebook này cho phép chạy 3 mô hình mà nhóm đề xuất: Chrono-C, Chrono-G và Chrono-R
- Thời gian chạy của cả 3 mô hình đều tối đa 2 tiếng, chúng tôi khuyến khích nên sử dụng cơ chế chạy offline trên Kaggle.
- Cell code 9 là cell cần phải chú ý, trước khi chạy bạn phải:
    - Thay đổi đường dẫn đến file data (Do nó thay đổi tùy theo tài khoản Kaggle)
    - Chọn loại mô hình muốn chạy

In [ ]:
!git clone https://github.com/CryAndRRich/chronolens.git

In [ ]:
%cd /kaggle/working/chronolens
CHRONO_PATH = "/kaggle/working/chronolens"

In [ ]:
!pip install -r requirements.txt

In [3]:
import sys

if CHRONO_PATH not in sys.path:
    sys.path.append(CHRONO_PATH)

In [4]:
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader

In [5]:
from config import *
from preprocess import *
from model import *
from utils import *

In [ ]:
RANDOM_SEED = 42
set_seed(RANDOM_SEED)

data_generator = torch.Generator()
data_generator.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [7]:
INPUT_ROOT = "YOUR_INPUT_PATH"
WORK_DIR = "/kaggle/working"

X_TRAIN = f"{INPUT_ROOT}/X_train.csv"
X_VAL = f"{INPUT_ROOT}/X_val.csv"
X_TEST = f"{INPUT_ROOT}/X_test.csv"
Y_TRAIN = f"{INPUT_ROOT}/Y_train.csv"
Y_VAL = f"{INPUT_ROOT}/Y_val.csv"

SUBMISSION_FILE = f"{WORK_DIR}/submission.csv"
CHECKPOINT_FILE = f"{WORK_DIR}/best_model.pth"
RETRAIN_CHECKPOINT_FILE = f"{WORK_DIR}/best_model_retrain.pth"

# MODEL_NAME nhận một trong ba giá trị: "chrono_r", "chrono_c" hoặc "chrono_g"
MODEL_NAME = "chrono_r"

In [8]:
data_manager = DataManager(
    input_root = INPUT_ROOT, 
    work_dir = WORK_DIR, 
    config_data = CONFIG_DATA,
    seed_worker = seed_worker,
    data_generator = data_generator,
    random_seed = RANDOM_SEED
)

In [9]:
x_train, y_train, x_val, y_val, x_test = data_manager.get_data()
train_loader, val_loader, test_loader = data_manager.get_dataloaders()
y_true = (y_val[CONFIG_DATA.ATTRIBUTE_COLS].values)[:, [1, 2, 3, 5, 6, 7]] 

In [ ]:
best_epoch = train_model(
    model_name=MODEL_NAME,
    data_manager=data_manager,
    train_loader=train_loader, 
    val_loader=val_loader,  
    y_val=y_true, 
    num_epochs=CONFIG_MODEL.NUM_EPOCHS,
    early_stopping=CONFIG_MODEL.EARLY_STOPPING,
    checkpoint_file=CHECKPOINT_FILE,
    device=DEVICE,
    verbose=True
)

In [11]:
model_kwargs = update_model_kwargs(
    data=data_manager,
    model_kwargs=CONFIG_MODEL.MODEL_KWARGS[MODEL_NAME]
)
model = get_model(
    name=MODEL_NAME,
    **model_kwargs  
).to(DEVICE)

model.load_state_dict(torch.load(CHECKPOINT_FILE, map_location=DEVICE))

model.eval()
val_pred = run_inference(model, val_loader, DEVICE)

In [ ]:
get_stats(y_true, val_pred)

In [13]:
x_full = pd.concat([x_train, x_val], ignore_index=True)
y_full = pd.concat([y_train, y_val], ignore_index=True)

full_dataset = UserBehaviorDataset(
    x_df=x_full, 
    y_df=y_full, 
    feature_cols=data_manager.FEATURE_COLS,
    attr_cols=data_manager.ATTRIBUTE_COLS,
    augment=True
)
full_loader = DataLoader(
    full_dataset, 
    batch_size=1024, 
    shuffle=True, 
    num_workers=data_manager.NUM_WORKERS, 
    worker_init_fn=seed_worker, 
    generator=data_generator,
    pin_memory=True, 
    drop_last=True, 
    persistent_workers=True  
)

In [ ]:
model = retrain_model(
    model_name=MODEL_NAME,
    data_manager=data_manager, 
    data_loader=full_loader,
    num_epochs=max(1, int(best_epoch * 1.1)),
    checkpoint_file=RETRAIN_CHECKPOINT_FILE,
    device=DEVICE,
    verbose=True
)

In [ ]:
model.eval()
test_preds = run_inference(model, test_loader, DEVICE)
submission_df = pd.DataFrame({"id": x_test["id"]})

for idx, attr in enumerate(data_manager.ATTRIBUTE_LIST):
    submission_df[f"attr_{attr}"] = test_preds[:, idx].astype(np.uint16)
    
submission_df.to_csv(SUBMISSION_FILE, index=False)
print(submission_df)
print(submission_df.dtypes)